# Multi-Scale CNN-BiLSTM Autoencoder (MSCNN-BiLSTM-AE) for NIDS

Notebook ini adalah implementasi **SOTA (State-of-the-Art)** untuk deteksi intrusi jaringan menggunakan pendekatan **Unsupervised Learning**.

## Tujuan
Mendeteksi serangan *Zero-Day* pada dataset **CSE-CIC-IDS2018** setelah dilatih HANYA pada trafik normal **CIC-IDS2017**.

## Arsitektur Model
- **Multi-Scale CNN:** Menangkap pola spasial jangka pendek, menengah, dan panjang (Kernel 3, 5, 7).
- **Bi-Directional LSTM:** Menangkap konteks temporal dua arah.
- **Attention Mechanism:** Fokus pada fitur paling signifikan.
- **Autoencoder:** Mendeteksi anomali berdasarkan *reconstruction error*.


## Colab Bootstrap (Colab-only)

Section ini hanya aktif saat runtime Google Colab.
Fungsinya: mount Drive, clone/pull branch target, optional symlink data dari Drive, setup `kaggle.json`,
dan symlink semua output artifact (`data/research`, `models/research`, `results/research`, `reports/research`) ke Drive.


In [ ]:
# Colab bootstrap (Colab-only): mount drive + clone/pull repo + data/output links + kaggle token.
from pathlib import Path
import os
import shutil
import subprocess
import sys

COLAB_BOOTSTRAP_ENABLE = True
COLAB_REPO_URL = "https://github.com/akwancakra/nids-mscnn-bilstm-autoencoder.git"
COLAB_BRANCH = "master"
COLAB_REPO_DIR = Path("/content/nids-mscnn-bilstm-autoencoder")

# Configurable Drive Paths
COLAB_DRIVE_MOUNT = Path("/content/drive")
COLAB_DRIVE_ROOT = COLAB_DRIVE_MOUNT / "MyDrive"
# Project Folder in Drive (Persistent Storage)
COLAB_PROJECT_DRIVE_ROOT = COLAB_DRIVE_ROOT / "nids-mscnn-bilstm-autoencoder"

# Input Data (Raw)
COLAB_RAW_DATA_DRIVE = COLAB_DRIVE_ROOT / "nids-data" / "raw"

# Flags
COLAB_LINK_RAW_FROM_DRIVE = True
COLAB_LINK_OUTPUTS_TO_DRIVE = True
COLAB_FORCE_RELINK_OUTPUTS = True

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _run_shell(cmd: list[str], cwd: Path | None = None) -> None:
    print("[CMD]", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def _mount_drive() -> None:
    from google.colab import drive
    drive.mount(str(COLAB_DRIVE_MOUNT))

def _clone_or_pull_repo() -> None:
    if not COLAB_REPO_DIR.exists():
        _run_shell(["git", "clone", COLAB_REPO_URL, str(COLAB_REPO_DIR)], cwd=Path("/content"))
    else:
        _run_shell(["git", "fetch", "--all"], cwd=COLAB_REPO_DIR)
    
    try:
        _run_shell(["git", "checkout", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
    except Exception:
        pass
        
    _run_shell(["git", "pull", "origin", COLAB_BRANCH], cwd=COLAB_REPO_DIR)

def _ensure_symlink_dir(repo_path: Path, drive_target: Path, allow_replace_existing: bool = False) -> None:
    """
    Robust symlink creation.
    """
    drive_target.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        current_target = repo_path.resolve()
        if current_target == drive_target.resolve():
            print(f"[INFO] Symlink OK: {repo_path} -> {drive_target}")
            return
        repo_path.unlink(missing_ok=True)

    elif repo_path.exists():
        if allow_replace_existing:
            if repo_path.is_dir():
                shutil.rmtree(repo_path, ignore_errors=True)
            else:
                repo_path.unlink(missing_ok=True)
        else:
            try:
                has_contents = any(repo_path.iterdir())
            except Exception:
                has_contents = True
            
            if has_contents:
                 print(f"[WARN] Path exists and is not symlink: {repo_path}. Skipping link (not empty).")
                 return
            
            if repo_path.is_dir():
                repo_path.rmdir()
            else:
                repo_path.unlink(missing_ok=True)

    repo_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(drive_target), str(repo_path), target_is_directory=True)
    print(f"[INFO] Symlink created: {repo_path} -> {drive_target}")

def _link_outputs_to_drive() -> None:
    # Map project folders to Drive folders
    # FLATTENED STRUCTURE: We link directly to repo root
    
    # Repo Root: /content/nids-mscnn-bilstm-autoencoder
    base_repo_project = COLAB_REPO_DIR 
    
    # Drive Root: /content/drive/MyDrive/nids-mscnn-bilstm-autoencoder
    base_drive_project = COLAB_PROJECT_DRIVE_ROOT
    
    mapping = {
        # Persist Processed Data DIRECTLY to Repo Root/data/processed
        base_repo_project / "data" / "processed": base_drive_project / "data" / "processed",
        
        # Persist Models
        base_repo_project / "models": base_drive_project / "models",
        
        # Persist Outputs/Logs
        base_repo_project / "output": base_drive_project / "output",
    }
    
    for repo_path, drive_target in mapping.items():
        _ensure_symlink_dir(
            repo_path,
            drive_target,
            allow_replace_existing=COLAB_FORCE_RELINK_OUTPUTS,
        )

IS_COLAB_RUNTIME = _is_colab_runtime()
print(f"IS_COLAB_RUNTIME={IS_COLAB_RUNTIME}")

if IS_COLAB_RUNTIME and COLAB_BOOTSTRAP_ENABLE:
    _mount_drive()
    _clone_or_pull_repo()
    
    if COLAB_LINK_OUTPUTS_TO_DRIVE:
        _link_outputs_to_drive()

    # Change working directory to REPO ROOT (not the inner folder)
    if COLAB_REPO_DIR.exists():
        os.chdir(COLAB_REPO_DIR)
        print(f"[INFO] Colab bootstrap done. cwd={Path.cwd()}")
    else:
        print(f"[WARN] Folder {COLAB_REPO_DIR} tidak ditemukan.")
else:
    print("[SKIP] Colab bootstrap is disabled or not running in Colab.")

In [ ]:
# Resolve project root (works for local + Colab), lalu optional install dependency.
from pathlib import Path
import os
import subprocess
import sys

CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "IntrusionDetectionSystem", # Check inner folder too
    Path.cwd().parent,
    Path("/content/nids-mscnn-bilstm-autoencoder"),
]

PROJECT_ROOT = None
for c in CANDIDATES:
    # We look for Python folder inside IntrusionDetectionSystem if we are at repo root
    if (c / "IntrusionDetectionSystem" / "Python").exists():
         PROJECT_ROOT = c / "IntrusionDetectionSystem"
         break
    # Or if we are already inside IntrusionDetectionSystem
    if (c / "Python").exists() and (c / "config.yaml").exists():
        PROJECT_ROOT = c.resolve()
        break

if PROJECT_ROOT is None:
    # Fallback: Assume we are at repo root and create structure if missing (virtual)
    print("Warning: Standard Project Root not found. Using current dir as root.")
    PROJECT_ROOT = Path.cwd()

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

# Add Python folder to sys.path
# If we are at repo root, the python code is in IntrusionDetectionSystem/Python
python_path = PROJECT_ROOT / 'Python'
if not python_path.exists() and (PROJECT_ROOT / 'IntrusionDetectionSystem' / 'Python').exists():
    python_path = PROJECT_ROOT / 'IntrusionDetectionSystem' / 'Python'
    
sys.path.append(str(python_path))
print(f"Added to sys.path: {python_path}")

INSTALL_DEPS = True
if INSTALL_DEPS:
    print("Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "tensorflow", "pandas", "numpy", "matplotlib", "scikit-learn", "pyyaml", "joblib", "seaborn", "tqdm"], check=True)
    print("Dependencies installed.")
else:
    print("INSTALL_DEPS = False (skip install).")

In [ ]:
# @title Import Libraries & Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import yaml
import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import seaborn as sns
from tqdm.auto import tqdm
# import tensorflow_addons as tfa # Removed due to incompatibility

# Import modul custom yang sudah kita buat
from models.mscnn_bilstm_ae import build_multiscale_cnn_bilstm_ae
from preprocessing.data_loader_npz import load_npz_data
from utils.thresholding import calculate_reconstruction_error, get_threshold_percentile, plot_error_distribution

print("Modules imported successfully!")

## 1. Data Preprocessing (New Pipeline)
Jalankan cell ini jika data processed belum tersedia. Script ini akan membaca RAW data dari Drive, melakukan cleaning, scaling (MinMax 0-1), dan sequencing, lalu menyimpannya ke folder processed project ini (yang tersimpan di Google Drive).

In [ ]:
# @title Run Data Preprocessing
# FORCE RE-RUN PREPROCESSING TO FIX LABEL BUG
RUN_PREPROCESSING = True 

# Determine Raw Data Root
RAW_DATA_ROOT = None

if 'COLAB_RAW_DATA_DRIVE' in locals():
    # If using the bootstrap cell variable
    RAW_DATA_ROOT = COLAB_RAW_DATA_DRIVE
elif 'COLAB_RAW_DATA_ROOT' in locals():
    # Fallback to old variable name if defined manually
    RAW_DATA_ROOT = COLAB_RAW_DATA_ROOT
elif (PROJECT_ROOT / "data" / "raw").exists():
    # Local fallback
    RAW_DATA_ROOT = PROJECT_ROOT / "data" / "raw"
    print(f"Using Local Raw Data: {RAW_DATA_ROOT}")

if RUN_PREPROCESSING and RAW_DATA_ROOT is not None:
    RAW_TRAIN_PATH = RAW_DATA_ROOT / "CIC-IDS2017"
    RAW_TEST_PATH = RAW_DATA_ROOT / "CSE-CIC-IDS2018"
    
    # OUTPUT KE ROOT REPO / data / processed (Symlinked)
    OUTPUT_DIR = Path("/content/nids-mscnn-bilstm-autoencoder/data/processed")
    
    # If not in Colab, use local project processed dir
    if not str(OUTPUT_DIR).startswith("/content") or not Path("/content").exists():
         OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

    print(f"Raw Train Path: {RAW_TRAIN_PATH}")
    print(f"Output Dir: {OUTPUT_DIR}")
    
    # Pastikan output dir ada
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Path ke script python
    script_path = PROJECT_ROOT / "Python" / "preprocessing" / "generate_sota_data.py"
    if not script_path.exists():
         # Coba path alternatif Colab structure
         script_path = Path("/content/nids-mscnn-bilstm-autoencoder/IntrusionDetectionSystem/Python/preprocessing/generate_sota_data.py")
    
    if RAW_TRAIN_PATH.exists():
        # Jalankan script preprocessing
        !python "{script_path}" \
            --raw_train "{RAW_TRAIN_PATH}" \
            --raw_test "{RAW_TEST_PATH}" \
            --output_dir "{OUTPUT_DIR}" \
            --seq_len 10 \
            --stride 1
    else:
        print(f"Raw data not found at {RAW_TRAIN_PATH}. Skipping preprocessing.")
else:
    print("Skipping preprocessing step (RUN_PREPROCESSING=False or Raw Data Not Found).")


In [ ]:
# @title QA Check: Verify Processed Data
# Jalankan cell ini untuk memastikan data hasil preprocessing sudah benar (Range 0-1, Tidak ada NaN)

def check_processed_data(data_path):
    if not data_path.exists():
        print(f"Directory not found: {data_path}")
        # Cek apakah symlink?
        if data_path.is_symlink():
            print(f"Symlink target: {data_path.resolve()}")
            if not data_path.resolve().exists():
                print("Symlink target BROKEN (not found).")
        return
        
    files = sorted(glob.glob(os.path.join(data_path, "*.npz")))
    if not files:
        print(f"No files found in {data_path}")
        return
        
    print(f"Checking random shard from: {data_path}")
    # Load random shard
    sample_file = files[0]
    data = np.load(sample_file)
    X = data['X']
    y = data['y']
    
    print(f"File: {os.path.basename(sample_file)}")
    print(f"Shape X: {X.shape}")
    print(f"Shape y: {y.shape}")
    print(f"Min Value: {np.min(X):.4f} (Should be >= 0.0)")
    print(f"Max Value: {np.max(X):.4f} (Should be <= 1.0)")
    print(f"Mean Value: {np.mean(X):.4f}")
    print(f"Has NaN: {np.isnan(X).any()}")
    
    # VERIFY LABEL DISTRIBUTION
    unique, counts = np.unique(y, return_counts=True)
    dist = dict(zip(unique, counts))
    print(f"Label Distribution: {dist}")
    if "test" in str(data_path).lower():
        if 0 in dist and 1 in dist:
            print("✅ SUCCESS: Both Benign (0) and Attack (1) are present in Test Set!")
        else:
            print("❌ ERROR: Missing classes in Test Set. Check preprocessing logic.")
    
    print("-"*30)

if 'PROJECT_ROOT' in locals():
    processed_root = PROJECT_ROOT / "data" / "processed"
    print(f"Checking Processed Root: {processed_root}")
    
    if processed_root.exists():
        print("--- TRAIN DATA CHECK ---")
        check_processed_data(processed_root / "train")
        print("\n--- TEST DATA CHECK ---")
        check_processed_data(processed_root / "test")
    else:
        print(f"Processed data directory not found at: {processed_root}")
        if processed_root.is_symlink():
            print(f"It is a symlink pointing to: {processed_root.resolve()}")
            print("Check if your Google Drive path is correct and contains data.")
else:
    print("PROJECT_ROOT not defined. Run previous cells.")

## 2. Konfigurasi Eksperimen
Kita definisikan parameter training dan path data di sini. 
**NOTE:** Kita menggunakan shards data yang sudah diproses dari project utama (`nids-cnn-lstm-autoencoder/data/research`) untuk konsistensi data.

In [ ]:
# @title Configuration Parameters
config = {
    'paths': {
        # Gunakan path RELATIF terhadap PROJECT_ROOT
        'train_data': 'data/processed/train',
        'test_data': 'data/processed/test',
        'output_dir': 'output/mscnn_bilstm_ae_experiment',
        'model_save_path': 'models/mscnn_bilstm_ae_best.keras' # <--- UPDATED (.keras)
    },
    'model': {
        'sequence_length': 10,
        'n_features': None,
        'encoding_dim': 16,
        'learning_rate': 0.0005
    },
    'training': {
        'batch_size': 256,
        'epochs': 50,
        'validation_split': 0.1,
        'patience': 5
    },
    'thresholding': {
        'percentile': 95.0
    }
}

# Execution Flags
RUN_TRAINING = False        # Set True to train new model
LOAD_PRETRAINED = True    # Set True to load existing model (if RUN_TRAINING=False)

# Buat output directory
if 'PROJECT_ROOT' in locals():
    out_dir = PROJECT_ROOT / config['paths']['output_dir']
    model_dir = PROJECT_ROOT / os.path.dirname(config['paths']['model_save_path'])
    
    out_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    
    # Update config with absolute paths
    config['paths']['train_data'] = str(PROJECT_ROOT / config['paths']['train_data'])
    config['paths']['test_data'] = str(PROJECT_ROOT / config['paths']['test_data'])
    config['paths']['output_dir'] = str(out_dir)
    config['paths']['model_save_path'] = str(PROJECT_ROOT / config['paths']['model_save_path'])
    
    print(f"Config loaded. Output dir: {out_dir}")
    print(f"RUN_TRAINING: {RUN_TRAINING}, LOAD_PRETRAINED: {LOAD_PRETRAINED}")
else:
    print("PROJECT_ROOT not set. Please run previous cells.")

## 3. Data Inspection (EDA)
Menganalisis distribusi data mentah (RAW) atau processed yang tersedia.
Cell ini akan mencoba mendeteksi data secara otomatis di path standar.

In [ ]:
# @title Data Inspection & Class Distribution
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm
import numpy as np
import os
import glob
from pathlib import Path

# Default values to prevent NameError
train_files = []
test_files = []

# --- AUTO DETECT DATA PATHS ---
# Coba deteksi path data processed (hasil run sebelumnya)
try:
    base_processed = None
    if 'config' in locals() and 'paths' in config:
        if 'processed_data' in config['paths']:
            base_processed = PROJECT_ROOT / config['paths']['processed_data']
        elif 'train_data' in config['paths']:
            base_processed = Path(config['paths']['train_data']).parent

    if base_processed is None:
        candidates = [
            Path("data/processed"),
            Path("../data/processed"),
            Path("/content/nids-cnn-lstm-autoencoder/data/research/sprint3_upgrade/processed"),
            Path("/content/nids-mscnn-bilstm-autoencoder/data/processed")
        ]
        for p in candidates:
            if p.exists():
                base_processed = p
                break

    if base_processed:
        print(f"[INFO] Inspecting data at: {base_processed}")
        train_files = sorted(glob.glob(str(base_processed / "train" / "*.npz")))
        test_files = sorted(glob.glob(str(base_processed / "test" / "*.npz")))
        
        if not train_files:
             train_files = sorted(glob.glob(str(base_processed / "**" / "train" / "*.npz"), recursive=True))
        if not test_files:
             test_files = sorted(glob.glob(str(base_processed / "**" / "test" / "*.npz"), recursive=True))
    else:
        print("[WARN] Processed data directory not found yet. Run preprocessing first or check paths.")
        
except Exception as e:
    print(f"[ERROR] Path detection failed: {e}")

def analyze_distribution(files, set_name="Dataset"):
    print(f"\n--- Analyzing {set_name} ---")
    if not files:
        print("No files found.")
        return

    try:
        with np.load(files[0], allow_pickle=True) as data:
            print(f"Sample File: {os.path.basename(files[0])}")
            print(f"Keys: {list(data.keys())}")
            for k in data.keys():
                obj = data[k]
                print(f"  {k}: shape={obj.shape}, dtype={obj.dtype}")
    except Exception as e:
        print(f"Error reading sample file: {e}")
        return

    label_counts = Counter()
    total_samples = 0
    
    scan_limit = min(len(files), 100) 
    print(f"Scanning {scan_limit} files (sample) for label distribution...")
    
    for f in tqdm(files[:scan_limit], desc=f"Scanning {set_name}"):
        try:
            with np.load(f, allow_pickle=True) as data:
                y = data['y'] if 'y' in data else (data['Y'] if 'Y' in data else None)
                if y is not None:
                    if len(y.shape) > 0:
                        label_counts.update(y)
                    else:
                        label_counts[y.item()] += 1
                    total_samples += len(y) if len(y.shape) > 0 else 1
        except: pass
        
    scale_factor = len(files) / scan_limit
    estimated_total = int(total_samples * scale_factor)
    
    print(f"Scanned Samples: {total_samples}")
    print(f"Estimated Total Samples: {estimated_total}")
    print(f"Label Distribution (Scanned): {dict(label_counts)}")
    
    if label_counts:
        plt.figure(figsize=(10, 5))
        keys = list(label_counts.keys())
        vals = list(label_counts.values())
        sns.barplot(x=keys, y=vals)
        plt.title(f"Label Distribution (Sampled) - {set_name}")
        plt.xlabel("Label (0=Benign, 1=Attack)")
        plt.ylabel("Count")
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.show()

if train_files:
    analyze_distribution(train_files, "Train Set (Benign Expected)")
else:
    print("Train files not found. (Maybe preprocessing hasn't run yet?)")

if test_files:
    analyze_distribution(test_files, "Test Set (Mixed Expected)")
else:
    print("Test files not found.")


### 3.1 EDA Data Raw (Opsional)
Inspeksi singkat file CSV mentah untuk melihat kolom, missing values, dan distribusi label.

In [ ]:
import pandas as pd
from pathlib import Path
import glob
import os
from IPython.display import display

def pick_first_existing(paths):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists():
            return p
    return None

raw_train_dir = None
raw_test_dir = None

if 'COLAB_RAW_DATA_ROOT' in locals():
    raw_train_dir = Path(COLAB_RAW_DATA_ROOT) / "CIC-IDS2017"
    raw_test_dir = Path(COLAB_RAW_DATA_ROOT) / "CSE-CIC-IDS2018"

if raw_train_dir is None:
    raw_train_dir = pick_first_existing([
        "/content/drive/MyDrive/nids-data/raw/CIC-IDS2017",
        PROJECT_ROOT / "data" / "raw" / "CIC-IDS2017"
    ])
if raw_test_dir is None:
    raw_test_dir = pick_first_existing([
        "/content/drive/MyDrive/nids-data/raw/CSE-CIC-IDS2018",
        PROJECT_ROOT / "data" / "raw" / "CSE-CIC-IDS2018"
    ])

def inspect_raw_dir(raw_dir, name):
    print(f"\n--- RAW {name} ---")
    if raw_dir is None or not Path(raw_dir).exists():
        print("Raw folder not found.")
        return
    raw_dir = Path(raw_dir)
    files = sorted(glob.glob(str(raw_dir / "*.csv")))
    if not files:
        files = sorted(glob.glob(str(raw_dir / "**" / "*.csv"), recursive=True))
    if not files:
        print(f"No CSV files found at {raw_dir}")
        return
    print(f"Folder: {raw_dir}")
    print(f"Files: {len(files)}")
    sample_file = files[0]
    print(f"Sample file: {os.path.basename(sample_file)}")
    try:
        df = pd.read_csv(sample_file, nrows=20000)
    except Exception:
        df = pd.read_csv(sample_file, nrows=5000, engine="python")
    print(f"Shape: {df.shape}")
    display(df.head(5))
    print("Missing (top 10):")
    print(df.isna().sum().sort_values(ascending=False).head(10))
    label_col = "Label" if "Label" in df.columns else ("label" if "label" in df.columns else None)
    if label_col:
        print(f"Label distribution ({label_col}):")
        print(df[label_col].value_counts().head(20))

inspect_raw_dir(raw_train_dir, "CIC-IDS2017 (Train)")
inspect_raw_dir(raw_test_dir, "CSE-CIC-IDS2018 (Test)")


### 3.2 Ringkasan Label CIC vs CSE (Raw)
Menampilkan total benign/attack dan distribusi per serangan dari dataset mentah.

In [ ]:
import pandas as pd
from pathlib import Path
import glob
from IPython.display import display

def find_label_column(sample_file):
    cols = pd.read_csv(sample_file, nrows=0, low_memory=False).columns.tolist()
    clean_map = {str(c).strip().lower(): c for c in cols}
    return clean_map.get("label")

def count_labels_in_dir(raw_dir, dataset_name, chunksize=200000):
    print(f"\n--- LABEL SUMMARY {dataset_name} ---")
    if raw_dir is None or not Path(raw_dir).exists():
        print("Raw folder not found.")
        return None, None
    raw_dir = Path(raw_dir)
    files = sorted(glob.glob(str(raw_dir / "*.csv")))
    if not files:
        files = sorted(glob.glob(str(raw_dir / "**" / "*.csv"), recursive=True))
    if not files:
        print(f"No CSV files found at {raw_dir}")
        return None, None
    label_col = find_label_column(files[0])
    if not label_col:
        print("Label column not found.")
        return None, None
    counts = {}
    for f in files:
        for chunk in pd.read_csv(f, usecols=[label_col], chunksize=chunksize, low_memory=False):
            vc = chunk[label_col].fillna("UNKNOWN").astype(str).value_counts()
            for k, v in vc.items():
                counts[k] = counts.get(k, 0) + int(v)
    if not counts:
        print("No label counts found.")
        return None, None
    total = int(sum(counts.values()))
    benign_total = int(sum(v for k, v in counts.items() if str(k).strip().upper() == "BENIGN"))
    attack_total = int(total - benign_total)
    overall_df = pd.DataFrame([
        {"dataset": dataset_name, "benign": benign_total, "attack": attack_total, "total": total}
    ])
    attack_rows = [
        {"dataset": dataset_name, "label": k, "count": int(v)}
        for k, v in counts.items()
        if str(k).strip().upper() != "BENIGN"
    ]
    attack_df = (
        pd.DataFrame(attack_rows).sort_values(by="count", ascending=False)
        if attack_rows
        else pd.DataFrame(columns=["dataset", "label", "count"])
    )
    display(overall_df)
    display(attack_df)
    return overall_df, attack_df

count_labels_in_dir(raw_train_dir, "CIC-IDS2017 (Train)")
count_labels_in_dir(raw_test_dir, "CSE-CIC-IDS2018 (Test)")


In [ ]:
# @title 1. Data Pipeline (Flexible: Memory Efficient vs Speed Optimized)
# Pilih mode sesuai kapasitas RAM Colab Anda.

# --- KONFIGURASI MODE ---
PIPELINE_MODE = "SPEED_OPTIMIZED"  # Options: "MEMORY_EFFICIENT" (12GB RAM) or "SPEED_OPTIMIZED" (25GB+ RAM)
# -----------------------

import tensorflow as tf
import glob
import numpy as np
import os

print(f"[INFO] Pipeline Mode: {PIPELINE_MODE}")

def npz_generator(files):
    """
    Generator yang membaca file .npz dari list file satu per satu dan yield batch data.
    """
    if not files:
        # print(f"Warning: No files provided")
        return
        
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                # Handle keys
                X = data['X'] if 'X' in data else (data['x'] if 'x' in data else None)
                
                if X is None:
                    continue
                    
                # Yield per sample (tf.data will batch it later)
                for i in range(len(X)):
                    # Autoencoder: Input = Target (X, X)
                    yield X[i], X[i] # Target is X for AE
        except Exception as e:
            print(f"Error reading {f}: {e}")

def create_dataset(files, batch_size=256, shuffle=True, input_shape=(10, 77)):
    """
    Membuat tf.data.Dataset dari list file shards.
    """
    # Tentukan output signature
    output_signature = (
        tf.TensorSpec(shape=input_shape, dtype=tf.float32), # Input X
        tf.TensorSpec(shape=input_shape, dtype=tf.float32)  # Target X (Autoencoder)
    )
    
    dataset = tf.data.Dataset.from_generator(
        lambda: npz_generator(files),
        output_signature=output_signature
    )
    
    # --- LOGIKA MODE ---
    if PIPELINE_MODE == "SPEED_OPTIMIZED":
        # Cache ke RAM setelah pembacaan pertama.
        # Epoch 1: Lambat. Epoch 2+: Kilat.
        dataset = dataset.cache()
        shuffle_buffer = 50000 if shuffle else 1000
    else:
        # Memory Efficient (Streaming)
        shuffle_buffer = 10000 if shuffle else 1000
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=shuffle_buffer)
        
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Helper untuk menghitung steps_per_epoch
def count_samples(files):
    total = 0
    if not files:
        print("Warning: No files to count.")
        return 0
        
    print(f"Counting samples from {len(files)} files...")
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                key = 'X' if 'X' in data else 'x'
                if key in data:
                    total += data[key].shape[0]
        except: pass
    return total

print("Data Pipeline Ready.")

## 4. Load Data Training (CIC-IDS2017)
Data ini harus berupa trafik **Benign (Normal)** saja.

In [ ]:
if 'PROJECT_ROOT' in locals() and (PROJECT_ROOT / "data" / "processed" / "train").exists():
    if RUN_TRAINING:
        print("[Phase 2] Building & Training Model (Streaming Mode)...")

        TRAIN_DIR = config['paths']['train_data']
        BATCH_SIZE = config['training']['batch_size']
        
        # 1. Split Files for Train/Val (Supaya Val data tidak bocor & loading cepat)
        all_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.npz")))
        split_idx = int(len(all_files) * (1 - config['training']['validation_split']))
        train_files = all_files[:split_idx]
        val_files = all_files[split_idx:]
        
        # 2. Count samples separately
        train_samples = count_samples(train_files)
        val_samples = count_samples(val_files)
        print(f"Train Samples: {train_samples}")
        print(f"Val Samples: {val_samples}")
        
        if train_samples > 0:
            # 3. Create Datasets
            train_dataset = create_dataset(train_files, batch_size=BATCH_SIZE, shuffle=True)
            val_dataset = create_dataset(val_files, batch_size=BATCH_SIZE, shuffle=False)
            
            # IMPORTANT: Repeat train dataset for infinite epochs
            # Ini yang memperbaiki error "ran out of data"
            train_dataset = train_dataset.repeat()
            
            train_steps = train_samples // BATCH_SIZE
            val_steps = val_samples // BATCH_SIZE
            
            print(f"Train Steps: {train_steps}")
            print(f"Val Steps: {val_steps}")
            
            # 3. Build Model
            input_shape = (10, 77)
            print(f"Input Shape: {input_shape}")

            model = build_multiscale_cnn_bilstm_ae(input_shape, encoding_dim=config['model']['encoding_dim'])
            model.summary()

            # Callbacks Setup
            model_save_dir = os.path.dirname(config['paths']['model_save_path'])
            os.makedirs(model_save_dir, exist_ok=True)
            
            callbacks = [
                # 1. Early Stopping
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_loss', 
                    patience=config['training']['patience'], 
                    restore_best_weights=True,
                    verbose=1
                ),
                # 2. Best Model Checkpoint (.keras)
                tf.keras.callbacks.ModelCheckpoint(
                    filepath=os.path.join(model_save_dir, 'best_model.keras'), # <--- .keras
                    monitor='val_loss',
                    save_best_only=True,
                    verbose=1
                ),
                # 3. Periodic Checkpoint (.keras)
                tf.keras.callbacks.ModelCheckpoint(
                    filepath=os.path.join(model_save_dir, 'checkpoint_epoch_{epoch:02d}.keras'), # <--- .keras
                    save_freq=train_steps * 25, # Save tiap 25 epoch
                    verbose=1
                )
            ]

            # Train
            history = model.fit(
                train_dataset,
                epochs=config['training']['epochs'],
                steps_per_epoch=train_steps,
                validation_data=val_dataset,
                validation_steps=val_steps,
                callbacks=callbacks,
                verbose=1
            )

            # Save Final Model
            model.save(config['paths']['model_save_path'])
            print(f"Model saved to {config['paths']['model_save_path']}")

            # Plot Training History
            plt.figure(figsize=(10, 4))
            plt.plot(history.history['loss'], label='Train Loss')
            plt.plot(history.history['val_loss'], label='Val Loss')
            plt.title('Model Loss')
            plt.ylabel('Loss (MSE)')
            plt.xlabel('Epoch')
            plt.legend()
            plt.show()
        else:
            print("No training samples found.")
            
    elif LOAD_PRETRAINED:
        print("[Phase 2] Loading Pretrained Model...")
        model_path = config['paths']['model_save_path']
        # Check if best model exists (.keras priority)
        best_model_path = os.path.join(os.path.dirname(model_path), 'best_model.keras')
        
        if os.path.exists(best_model_path):
            target_path = best_model_path
        elif os.path.exists(model_path):
            target_path = model_path
        else:
            # Fallback check for old .h5
            best_model_h5 = os.path.join(os.path.dirname(model_path), 'best_model.h5')
            target_path = best_model_h5 if os.path.exists(best_model_h5) else model_path

        if os.path.exists(target_path):
            model = tf.keras.models.load_model(target_path)
            print(f"Model loaded from {target_path}")
            model.summary()
            
            # Re-prepare validation dataset for thresholding
            TRAIN_DIR = config['paths']['train_data']
            BATCH_SIZE = config['training']['batch_size']
            
            all_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.npz")))
            split_idx = int(len(all_files) * (1 - config['training']['validation_split']))
            val_files = all_files[split_idx:]
            
            val_samples = count_samples(val_files)
            val_dataset = create_dataset(val_files, batch_size=BATCH_SIZE, shuffle=False)
            val_steps = val_samples // BATCH_SIZE
            
            print("Validation dataset prepared for thresholding.")
        else:
            print(f"Model file not found at {target_path}")
    else:
        print("Skipping Training and Loading (RUN_TRAINING=False, LOAD_PRETRAINED=False).")
else:
    print("Train directory not found or Project Root not set.")

## 5. Determine Threshold
Menentukan ambang batas (threshold) untuk mendeteksi anomali berdasarkan error rekonstruksi pada data validasi.
Threshold ini dihitung ulang secara dinamis (biasanya P99.0 atau P99.9) setiap kali model dimuat.

In [ ]:
# @title 5. Determine Threshold & Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

def find_threshold(model, val_ds, percentile=95):
    print("Calculating reconstruction errors on Validation Set (Benign)...")
    val_errors = []
    
    # Calculate MSE for all validation samples
    # val_ds should be finite (not repeated)
    
    for batch_x, _ in tqdm(val_ds, desc="Calculating Threshold"):
        recon = model.predict(batch_x, verbose=0)
        # MSE per sample
        mse = np.mean(np.square(batch_x - recon), axis=(1, 2))
        val_errors.extend(mse)
    
    val_errors = np.array(val_errors)
    
    # Determine Threshold
    threshold_val = np.percentile(val_errors, percentile)
    print(f"\nConfiguration:")
    print(f"  percentile: {percentile}%")
    print(f"  threshold : {threshold_val:.6f}")
    
    # Plot Distribution
    plt.figure(figsize=(10, 5))
    sns.histplot(val_errors, bins=50, kde=True, color='blue', label='Benign (Val) Errors')
    plt.axvline(threshold_val, color='red', linestyle='--', label=f'Threshold ({percentile}%)')
    plt.title(f"Reconstruction Error Distribution (Benign) - Threshold: {threshold_val:.5f}")
    plt.xlabel("Mean Squared Error (MSE)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return threshold_val

# Execute
if 'val_dataset' in locals() and 'model' in locals():
    # Gunakan percentile dari config jika ada, default 95
    pct = config.get('thresholding', {}).get('percentile', 95)
    
    # Override logic: jika percentile > 99, mungkin terlalu strict untuk MSE
    # Kita pakai 95 sebagai default yang aman jika user belum tuning
    # pct = 95 
    
    threshold = find_threshold(model, val_dataset, percentile=pct)
    print(f"✅ Threshold set to: {threshold}")
else:
    print("❌ Validation dataset (val_dataset) or Model not found. Run training/loading first.")

## 6. Evaluation (Dual-Set)
Evaluasi model pada dua dataset:
1. **CSE-CIC-IDS2018 (Wajib):** Test set utama (Zero-Day Attack).
2. **CIC-IDS2017 (Opsional):** Test set in-domain (jika tersedia).

Set variable `RUN_ALL_EVAL = True` untuk menjalankan keduanya.

In [ ]:
# @title Dual Evaluation (CSE + CIC) - Fast Mode
RUN_ALL_EVAL = True

# --- CONFIG: EVALUATION SPEED ---
EVAL_SAMPLE_FRACTION = 0.2  # 0.2 = Run on 20% of data (Faster). 1.0 = Full Data.
# --------------------------------

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import glob
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm.notebook import tqdm

def evaluate_performance(y_true, y_pred, name="Test Set", color='Blues'):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"\n[{name}] Performance:")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=color)
    plt.title(f'Confusion Matrix ({name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    return f1

def npz_generator_with_label(files):
    for f in files:
        try:
            with np.load(f, allow_pickle=True) as data:
                X = data['X'] if 'X' in data else (data['x'] if 'x' in data else None)
                y = data['y'] if 'y' in data else (data['Y'] if 'Y' in data else None)
                
                if X is not None and y is not None:
                    for i in range(len(X)):
                        yield X[i], y[i]
        except Exception: pass

def create_eval_dataset(files, batch_size=256):
    # Specialized dataset for evaluation returning (X, y) instead of (X, X)
    output_signature = (
        tf.TensorSpec(shape=(10, 77), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32) # Label is scalar int
    )
    
    dataset = tf.data.Dataset.from_generator(
        lambda: npz_generator_with_label(files),
        output_signature=output_signature
    )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def calculate_errors_and_predict(dataset, model, threshold, steps=None):
    all_errors = []
    all_y_true = []
    
    desc = "Predicting"
    prog_bar = tqdm(dataset, total=steps, desc=desc) if steps else tqdm(dataset, desc=desc)
    
    # Counter untuk limit sample
    batch_count = 0
    
    for batch_x, batch_y in prog_bar:
        # Predict Reconstruction
        recon = model.predict(batch_x, verbose=0)
        # Calculate MSE per sample
        mse = np.mean(np.square(batch_x - recon), axis=(1, 2))
        
        all_errors.extend(mse)
        all_y_true.extend(batch_y.numpy())
        
        # Stop jika sudah mencapai steps (untuk fast mode)
        batch_count += 1
        if steps and batch_count >= steps:
            break
    
    all_errors = np.array(all_errors)
    all_y_true = np.array(all_y_true)
    
    # Thresholding
    y_pred = (all_errors > threshold).astype(int)
    
    return all_y_true, y_pred, all_errors

# --- 1. Evaluate on CSE-CIC-IDS2018 (Main Test Set) ---
print("="*40)
print("1. EVALUATION: CSE-CIC-IDS2018 (Cross-Domain / Zero-Day)")
print("="*40)

if 'threshold' not in locals():
    print("⚠️ Threshold not defined. Using default 0.05")
    threshold = 0.05

# FORCE RELOAD CSE DATASET in (X, y) mode
cse_ds = None
cse_steps = None
test_files = []

# Locate files
if 'test_dir' in locals() and test_dir.exists():
     test_files = sorted(glob.glob(str(test_dir / "*.npz")))
elif 'config' in locals():
     base_test = PROJECT_ROOT / config['paths']['test_data']
     if base_test.exists():
         test_files = sorted(glob.glob(str(base_test / "*.npz")))

# Fallback Search
if not test_files:
    candidates = [
        PROJECT_ROOT / "data/processed/test",
        Path("/content/nids-mscnn-bilstm-autoencoder/data/processed/test"),
        Path("/content/nids-cnn-lstm-autoencoder/data/research/sprint3_upgrade/processed/test")
    ]
    for c in candidates:
        if c.exists():
            test_files = sorted(glob.glob(str(c / "*.npz")))
            if test_files: break

if test_files:
    # --- FAST MODE LOGIC ---
    total_samples = 0
    # Simple count if count_samples not available
    try:
        total_samples = count_samples(test_files)
    except NameError:
        print("Counting samples manually...")
        for f in test_files[:5]: # Estimasi dari 5 file pertama biar cepet
            try: with np.load(f) as d: total_samples += d['y'].shape[0]
            except: pass
        total_samples = total_samples * (len(test_files) // 5) if len(test_files) > 5 else total_samples

    BATCH_SIZE = config['training']['batch_size'] if 'config' in locals() else 256
    
    full_steps = max(1, total_samples // BATCH_SIZE)
    # Apply Sample Fraction
    cse_steps = int(full_steps * EVAL_SAMPLE_FRACTION)
    cse_steps = max(1, cse_steps) # Minimal 1 step
    
    print(f"Found {len(test_files)} files. Est. Total Samples: {total_samples}")
    print(f"Running FAST EVALUATION on {EVAL_SAMPLE_FRACTION*100}% Data.")
    print(f"Processing {cse_steps} batches (approx {cse_steps*BATCH_SIZE} samples)...")
    
    # Create DATASET with LABELS specifically for evaluation
    cse_ds = create_eval_dataset(test_files, batch_size=BATCH_SIZE)
    
    y_true_cse, y_pred_cse, errors_cse = calculate_errors_and_predict(cse_ds, model, threshold, cse_steps)
    evaluate_performance(y_true_cse, y_pred_cse, name="CSE-CIC-IDS2018", color='Blues')
else:
    print("❌ CSE-CIC-IDS2018 Dataset files not found.")


# --- 2. Evaluate on CIC-IDS2017 (In-Domain Test Set) ---
if RUN_ALL_EVAL:
    print("\n" + "="*40)
    print("2. EVALUATION: CIC-IDS2017 (In-Domain Reference)")
    print("="*40)
    
    cic_test_path = None
    possible_roots = [
        PROJECT_ROOT / "data" / "processed",
        Path("/content/nids-mscnn-bilstm-autoencoder/data/processed"),
        Path("/content/nids-cnn-lstm-autoencoder/data/research/sprint3_upgrade/processed"),
    ]
    
    for root in possible_roots:
        candidate = root / "test_cic"
        if candidate.exists():
            cic_test_path = candidate
            print(f"Found CIC Test Path at: {candidate}")
            break
            
    if cic_test_path:
        cic_files = sorted(glob.glob(str(cic_test_path / "*.npz")))
        if cic_files:
            try:
                 total_samples_cic = count_samples(cic_files)
            except NameError:
                 total_samples_cic = 0 # Skip detail count
                 
            full_steps_cic = max(1, total_samples_cic // BATCH_SIZE)
            
            # Apply Sample Fraction
            cic_steps = int(full_steps_cic * EVAL_SAMPLE_FRACTION)
            cic_steps = max(1, cic_steps)

            print(f"Running FAST EVALUATION on {EVAL_SAMPLE_FRACTION*100}% CIC Data.")
            print(f"Processing {cic_steps} batches (approx {cic_steps*BATCH_SIZE} samples)...")
            
            # Create Dataset
            cic_ds = create_eval_dataset(cic_files, batch_size=BATCH_SIZE)
            
            y_true_cic, y_pred_cic, errors_cic = calculate_errors_and_predict(cic_ds, model, threshold, cic_steps)
            evaluate_performance(y_true_cic, y_pred_cic, name="CIC-IDS2017", color='Greens')
        else:
            print("❌ CIC Test folder empty.")
    else:
        print("ℹ️ CIC-IDS2017 Test Data folder 'test_cic' not found.")

## 7. Comprehensive Experiment Summary & Review
Ringkasan lengkap hasil eksperimen untuk evaluasi mandiri dan reviewer.
Mencakup metrik kunci, interpretasi performa, dan catatan anomali.

In [ ]:
# @title Display Final Evaluation Results
import yaml
import pandas as pd
from IPython.display import display, Markdown

result_path = os.path.join(config['paths']['output_dir'], 'evaluation_results.yaml')

if os.path.exists(result_path):
    with open(result_path, 'r') as f:
        results = yaml.safe_load(f)
    
    print("\n" + "="*40)
    print("FINAL EXPERIMENT RESULTS")
    print("="*40)
    
    # Convert to DataFrame for nicer display
    metrics_df = pd.DataFrame([results])
    # Reorder columns if needed or just display
    display(metrics_df[['threshold', 'accuracy', 'precision', 'recall', 'f1_score', 'auc']])
    
    print("\nConfusion Matrix:")
    print(np.array(results['confusion_matrix']))
    
    # Optional: Display as Markdown table
    md_table = f"""
| Metric | Value |
| :--- | :--- |
| **Threshold** | `{results['threshold']:.6f}` |
| **Accuracy** | `{results['accuracy']:.4f}` |
| **Precision** | `{results['precision']:.4f}` |
| **Recall** | `{results['recall']:.4f}` |
| **F1-Score** | `{results['f1_score']:.4f}` |
| **AUC-ROC** | `{results['auc']:.4f}` |
"""
    display(Markdown(md_table))

else:
    print(f"Results file not found at {result_path}. Make sure the evaluation step completed successfully.")